## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
df=pd.read_csv('fake_or_real_news.csv')

## 3. Data Cleaning

In [ ]:
df.columns

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')

In [ ]:
df=df.drop(columns='Unnamed: 0')

In [ ]:
df.shape

(6335, 3)

In [ ]:
df.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   6335 non-null   object
 1   text    6335 non-null   object
 2   label   6335 non-null   object
dtypes: object(3)
memory usage: 148.6+ KB


In [ ]:
df.isnull().sum()

,0
title,0
text,0
label,0


In [ ]:
df[['title','text']].apply(lambda x:x.str.strip().eq("").sum())

,0
title,0
text,36


In [ ]:
df[df['text'].str.strip().eq("")]

,title,text,label
106,The Arcturian Group by Marilyn Raffaele Octobe...,,FAKE
710,MARKETWATCH LEFTIST: MSM’s “Blatant” Anti Trum...,,FAKE
806,Southern Poverty Law Center Targets Anti-Jihad...,,FAKE
919,Refugee Resettlement Watch: Swept Away In Nort...,,FAKE
940,Michael Bloomberg Names Technological Unemploy...,,FAKE
1664,Alert News : Putins Army Is Coming For World W...,,FAKE
1736,An LDS Reader Takes A Look At Trump Accuser Je...,,FAKE
1851,America’s Senator Jeff Sessions Warns of Worse...,,FAKE
1883,Paris Migrant Campers Increase after Calais Is...,,FAKE
1941,Putins Army is coming for World war 3 against ...,,FAKE


In [ ]:
df.drop(df[df['text'].str.strip().eq("")].index,inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(29)

In [ ]:
df.drop_duplicates(inplace=True)

## 4. Text Preprocessing

In [ ]:
df['context']=df['title']+' '+df['text']

In [ ]:
df['label'].value_counts()

,count
label,
REAL,3154
FAKE,3116


In [ ]:
df['context']=df['context'].apply(lambda x: x.lower())

In [ ]:
import re
df['context']=df['context'].apply(lambda x: re.sub(r"http\S+|www\S+","",x))

In [ ]:
df['context']=df['context'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]',"",x))

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize
df['context']=df['context'].apply(word_tokenize)

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from nltk.corpus import stopwords
stop_words=stopwords.words('english')

In [ ]:
def remove_stop_words(txt):
  t=[]
  for i in txt:
    if i not in stop_words:
      t.append(i)
  return ' '.join(t)
df['context']=df['context'].apply(remove_stop_words)


In [ ]:
from nltk.stem import WordNetLemmatizer
lemma=WordNetLemmatizer()
def lemmatize_text(txt):
  word=word_tokenize(txt)
  t=[]
  for i in word:
    t.append(lemma.lemmatize(i))
  return " ".join(t)

df['context']=df['context'].apply(lemmatize_text)


In [ ]:
df.head()

,title,text,label,context
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,smell hillary fear daniel greenfield shillman ...
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,watch exact moment paul ryan committed politic...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,kerry go paris gesture sympathy u secretary st...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE,bernie supporter twitter erupt anger dnc tried...
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL,battle new york primary matter primary day new...


In [ ]:
def preprocess_text(txt):
  txt=txt.lower()
  txt=re.sub(r"http\S+|www\S+","",txt)
  txt=re.sub(r'[^a-zA-Z0-9\s]',"",txt)
  # txt=word_tokenize(txt)    --->(use this inside only if you not defined it outside)
  # stop_words=stopwords.words('english')
  txt=[i for i in txt if i not in stop_words]
  # lemma=WordNetLemmatizer()
  txt=[lemma.lemmatize(i) for i in txt]
  return " ".join(txt)

## 5. Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['label']=le.fit_transform(df['label'])

In [ ]:
df['label'].value_counts()

,count
label,
1,3154
0,3116


## 6. Train-Test Split

In [ ]:
X=df['context']
y=df['label']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 7. Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

### Bag of Words

In [ ]:
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train)
X_test_bow=bow.transform(X_test)

In [ ]:
X_train_bow.shape, X_test_bow.shape

((5016, 70930), (1254, 70930))

### TF-IDF

In [ ]:
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

In [ ]:
X_train_tfidf.shape, X_test_tfidf.shape

((5016, 70930), (1254, 70930))

## 8. Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
lg=LogisticRegression()
lg.fit(X_train_bow, y_train)

LogisticRegression()

In [ ]:
print(f'accuracy of lg-bow= {accuracy_score(y_test,lg.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,lg.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,lg.predict(X_test_bow))}')
print(f'confusion matrix lg-bow=\n{confusion_matrix(y_test,lg.predict(X_test_bow))}')

accuracy of lg-bow= 0.9202551834130781
accuracy on train=1.0
classification report= 
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       623
           1       0.93      0.91      0.92       631

    accuracy                           0.92      1254
   macro avg       0.92      0.92      0.92      1254
weighted avg       0.92      0.92      0.92      1254

confusion matrix lg-bow=
[[578  45]
 [ 55 576]]


In [ ]:
lg2=LogisticRegression()
lg2.fit(X_train_tfidf,y_train)

LogisticRegression()

In [ ]:
print(f'accuracy of lg-tfidf= {accuracy_score(y_test,lg2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,lg2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,lg2.predict(X_test_tfidf))}')
print(f'confusion matrix lg-tfidf=\n{confusion_matrix(y_test,lg2.predict(X_test_tfidf))}')

accuracy of lg-tfidf= 0.9250398724082934
accuracy on train=0.95414673046252
classification report= 
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       623
           1       0.96      0.89      0.92       631

    accuracy                           0.93      1254
   macro avg       0.93      0.93      0.92      1254
weighted avg       0.93      0.93      0.92      1254

confusion matrix lg-tfidf=
[[599  24]
 [ 70 561]]


In [ ]:
svc=SVC()
svc.fit(X_train_bow,y_train)

SVC()

In [ ]:
print(f'accuracy of svc-bow= {accuracy_score(y_test,svc.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,svc.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,svc.predict(X_test_bow))}')
print(f'confusion matrix svc-bow=\n{confusion_matrix(y_test,svc.predict(X_test_bow))}')

accuracy of svc-bow= 0.8803827751196173
accuracy on train=0.9358054226475279
classification report= 
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       623
           1       0.94      0.81      0.87       631

    accuracy                           0.88      1254
   macro avg       0.89      0.88      0.88      1254
weighted avg       0.89      0.88      0.88      1254

confusion matrix svc-bow=
[[593  30]
 [120 511]]


In [ ]:
svc2=SVC()
svc2.fit(X_train_tfidf,y_train)

SVC()

In [ ]:
print(f'accuracy of svc-tfidf= {accuracy_score(y_test,svc2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,svc2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,svc2.predict(X_test_tfidf))}')
print(f'confusion matrix svc-tfidf=\n{confusion_matrix(y_test,svc2.predict(X_test_tfidf))}')

accuracy of svc-tfidf= 0.9370015948963317
accuracy on train=0.9978070175438597
classification report= 
              precision    recall  f1-score   support

           0       0.91      0.96      0.94       623
           1       0.96      0.91      0.94       631

    accuracy                           0.94      1254
   macro avg       0.94      0.94      0.94      1254
weighted avg       0.94      0.94      0.94      1254

confusion matrix svc-tfidf=
[[601  22]
 [ 57 574]]


In [ ]:
nb=MultinomialNB()
nb.fit(X_train_bow,y_train)

MultinomialNB()

In [ ]:
print(f'accuracy of nb-bow= {accuracy_score(y_test,nb.predict(X_test_bow))}')
print(f'accuracy on train={accuracy_score(y_train,nb.predict(X_train_bow))}')
print(f'classification report= \n{classification_report(y_test,nb.predict(X_test_bow))}')
print(f'confusion matrix nb-bow=\n{confusion_matrix(y_test,nb.predict(X_test_bow))}')

accuracy of nb-bow= 0.8899521531100478
accuracy on train=0.9479665071770335
classification report= 
              precision    recall  f1-score   support

           0       0.91      0.86      0.89       623
           1       0.87      0.92      0.89       631

    accuracy                           0.89      1254
   macro avg       0.89      0.89      0.89      1254
weighted avg       0.89      0.89      0.89      1254

confusion matrix nb-bow=
[[537  86]
 [ 52 579]]


In [ ]:
nb2=MultinomialNB()
nb2.fit(X_train_tfidf,y_train)

MultinomialNB()

In [ ]:
print(f'accuracy of nb-tfidf= {accuracy_score(y_test,nb2.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,nb2.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,nb2.predict(X_test_tfidf))}')
print(f'confusion matrix nb-tfidf=\n{confusion_matrix(y_test,nb2.predict(X_test_tfidf))}')

accuracy of nb-tfidf= 0.8301435406698564
accuracy on train=0.8903508771929824
classification report= 
              precision    recall  f1-score   support

           0       0.97      0.68      0.80       623
           1       0.76      0.98      0.85       631

    accuracy                           0.83      1254
   macro avg       0.86      0.83      0.83      1254
weighted avg       0.86      0.83      0.83      1254

confusion matrix nb-tfidf=
[[425 198]
 [ 15 616]]


In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression",
        "Multinomial Naive Bayes",
        "Multinomial Naive Bayes",
        "Linear SVM",
        "Linear SVM"
    ],
    "Feature Extraction": [
        "Bag of Words",
        "TF-IDF",
        "Bag of Words",
        "TF-IDF",
        "Bag of Words",
        "TF-IDF"
    ],
    "Train Accuracy": [
        1.0,
        0.9541,
        0.9480,
        0.8904,
        0.9358,
        0.9978
    ],
    "Test Accuracy": [
        0.9203,
        0.9250,
        0.8900,
        0.8301,
        0.8804,
        0.9370
    ]
})

results.sort_values(by="Test Accuracy", ascending=False)

,Model,Feature Extraction,Train Accuracy,Test Accuracy
5,Linear SVM,TF-IDF,0.9978,0.9370
1,Logistic Regression,TF-IDF,0.9541,0.9250
0,Logistic Regression,Bag of Words,1.0000,0.9203
2,Multinomial Naive Bayes,Bag of Words,0.9480,0.8900
4,Linear SVM,Bag of Words,0.9358,0.8804
3,Multinomial Naive Bayes,TF-IDF,0.8904,0.8301


## 9. Hyperparameter Tuning

In [ ]:
c_values=[0.01, 0.1, 1,2,5, 10,20]
for i in c_values:
  svc3=SVC(C=i)
  svc3.fit(X_train_tfidf,y_train)
  print(f"C={i}")
  print(f'accuracy of svc-tfidf= {accuracy_score(y_test,svc3.predict(X_test_tfidf))}')
  print(f'accuracy on train={accuracy_score(y_train,svc3.predict(X_train_tfidf))}')
  print(f'classification report= \n{classification_report(y_test,svc3.predict(X_test_tfidf))}')
  print(f'confusion matrix svc-tfidf=\n{confusion_matrix(y_test,svc3.predict(X_test_tfidf))}\n')

C=0.01
accuracy of svc-tfidf= 0.5031897926634769
accuracy on train=0.5029904306220095


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


classification report= 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       623
           1       0.50      1.00      0.67       631

    accuracy                           0.50      1254
   macro avg       0.25      0.50      0.33      1254
weighted avg       0.25      0.50      0.34      1254

confusion matrix svc-tfidf=
[[  0 623]
 [  0 631]]

C=0.1
accuracy of svc-tfidf= 0.7950558213716108
accuracy on train=0.8287480063795853
classification report= 
              precision    recall  f1-score   support

           0       0.72      0.97      0.83       623
           1       0.96      0.62      0.75       631

    accuracy                           0.80      1254
   macro avg       0.84      0.80      0.79      1254
weighted avg       0.84      0.80      0.79      1254

confusion matrix svc-tfidf=
[[607  16]
 [241 390]]

C=1
accuracy of svc-tfidf= 0.9370015948963317
accuracy on train=0.9978070175438597
classification report= 
  

In [ ]:
from sklearn.svm import LinearSVC
c_values=[0.01, 0.1, 1,2,5, 10,20]
for i in c_values:
  svc4=LinearSVC(C=i)
  svc4.fit(X_train_tfidf,y_train)
  print(f"C={i}")
  print(f'accuracy of linear svc-tfidf= {accuracy_score(y_test,svc4.predict(X_test_tfidf))}')
  print(f'accuracy on train={accuracy_score(y_train,svc4.predict(X_train_tfidf))}')
  print(f'classification report= \n{classification_report(y_test,svc4.predict(X_test_tfidf))}')
  print(f'confusion matrix linear svc-tfidf=\n{confusion_matrix(y_test,svc4.predict(X_test_tfidf))}\n')

C=0.01
accuracy of linear svc-tfidf= 0.868421052631579
accuracy on train=0.8787878787878788
classification report= 
              precision    recall  f1-score   support

           0       0.82      0.94      0.88       623
           1       0.93      0.80      0.86       631

    accuracy                           0.87      1254
   macro avg       0.88      0.87      0.87      1254
weighted avg       0.88      0.87      0.87      1254

confusion matrix linear svc-tfidf=
[[585  38]
 [127 504]]

C=0.1
accuracy of linear svc-tfidf= 0.9250398724082934
accuracy on train=0.9585326953748007
classification report= 
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       623
           1       0.96      0.89      0.92       631

    accuracy                           0.93      1254
   macro avg       0.93      0.93      0.92      1254
weighted avg       0.93      0.93      0.92      1254

confusion matrix linear svc-tfidf=
[[599  24]
 [ 70 561

## 10. Final Model

In [ ]:
final_model=SVC(C=2)
final_model.fit(X_train_tfidf,y_train)
print(f'accuracy of svc-tfidf= {accuracy_score(y_test,final_model.predict(X_test_tfidf))}')
print(f'accuracy on train={accuracy_score(y_train,final_model.predict(X_train_tfidf))}')
print(f'classification report= \n{classification_report(y_test,final_model.predict(X_test_tfidf))}')
print(f'confusion matrix svc-tfidf=\n{confusion_matrix(y_test,final_model.predict(X_test_tfidf))}\n')

accuracy of svc-tfidf= 0.9441786283891547
accuracy on train=1.0
classification report= 
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       623
           1       0.97      0.91      0.94       631

    accuracy                           0.94      1254
   macro avg       0.95      0.94      0.94      1254
weighted avg       0.95      0.94      0.94      1254

confusion matrix svc-tfidf=
[[607  16]
 [ 54 577]]



## 11. Save Model and Vectorizer

In [ ]:
import joblib

joblib.dump(tfidf,"tfidf_vectorizer.pkl")
joblib.dump(final_model,"final_model.pkl")

['final_model.pkl']

## 12. Test Prediction Pipeline

In [ ]:
loaded_tfidf = joblib.load("tfidf_vectorizer.pkl")
loaded_model = joblib.load("final_model.pkl")
sample_text = ["This is a sample news article for testing the model"]

sample_vector = loaded_tfidf.transform(sample_text)

prediction = loaded_model.predict(sample_vector)

print(prediction)

[0]


In [ ]:
sample_text = "Breaking News! Visit https://example.com for MORE information about the government's policies."

cleaned_text = preprocess_text(sample_text)

vectorized_text = loaded_tfidf.transform([cleaned_text])

prediction = loaded_model.predict(vectorized_text)

print("Prediction:", le.inverse_transform(prediction)[0])

Prediction: FAKE
